In [1]:
import pandas as pd 
import numpy as np 
import random 
import os 
import json 
import matplotlib.pyplot as plt 
import seaborn as sns 
from datetime import datetime, timedelta 
import funcoes

In [2]:
caminho_do_arquivo = '../dados/raw/vendas.csv'
df = pd.read_csv(caminho_do_arquivo)
df_original = df.copy()

In [3]:
# Visão geral do df
df_original

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-01-13,Cliente_024,Mouse,Periféricos,Norte,2.0,120.0
1,2,2024-08-04,Cliente_018,Notebook,Computadores,Sul,NaN,3500.0
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
3,4,2024-06-23,Cliente_013,Mouse,Periféricos,Sudeste,7.0,120.0
4,5,2024-11-05,Cliente_030,Tablet,Celulares,Centro-Oeste,6.0,1800.0
...,...,...,...,...,...,...,...,...
145,146,2024-08-09,Cliente_019,Notebook,Computadores,Norte,2.0,3500.0
146,147,2024-12-09,Cliente_028,Teclado,Periféricos,Sudeste,6.0,250.0
147,148,2024-06-08,Cliente_008,Tablet,Celulares,Sudeste,10.0,1800.0
148,149,2024-07-07,Cliente_018,Tablet,Celulares,Norte,9.0,1800.0


In [4]:
# Informações relevantes para investigar os dados
funcoes.analise_dados(df_original)

,Tipos de Dados,Qtd_Não_Nulos,Valores Únicos,Qtd_Nulos,% Nulos,Duplicados
id_venda,int64,150,150,0,0.0%,0
data_venda,object,150,113,0,0.0%,0
cliente,object,150,30,0,0.0%,0
produto,object,150,9,0,0.0%,0
categoria,object,150,3,0,0.0%,0
regiao,object,150,5,0,0.0%,0
quantidade,float64,144,10,6,4.0%,0
preco_unitario,float64,147,6,3,2.0%,0


In [5]:
# Inspecionando elementos
funcoes.contagem_valores(df_original)

--- RELATÓRIO DE VALORES ÚNICOS E CONTAGENS ---

[ ID_VENDA ] -> 150 valores. (Omitido)

[ DATA_VENDA ]
'2024-05-15' (10 letras): 5  |  'DATA INVALIDA' (13 letras): 4  |  '2024-11-22' (10 letras): 4
'2024-02-08' (10 letras): 3  |  '2024-10-26' (10 letras): 3  |  '2024-11-01' (10 letras): 2
'2024-07-21' (10 letras): 2  |  '2024-05-30' (10 letras): 2  |  '2024-09-16' (10 letras): 2
'2024-07-29' (10 letras): 2  |  '2024-05-05' (10 letras): 2  |  '2024-05-08' (10 letras): 2
'2024-02-25' (10 letras): 2  |  '2024-07-23' (10 letras): 2  |  '2024-10-22' (10 letras): 2
'2024-01-17' (10 letras): 2  |  '2024-11-05' (10 letras): 2  |  '2024-08-28' (10 letras): 2
'2024-08-08' (10 letras): 2  |  '2024-04-22' (10 letras): 2  |  '2024-12-03' (10 letras): 2
'2024-03-07' (10 letras): 2  |  '2024-03-20' (10 letras): 2  |  '2024-06-16' (10 letras): 2
'2024-07-04' (10 letras): 2  |  '2024-11-27' (10 letras): 2  |  '2024-05-27' (10 letras): 2
'2024-06-08' (10 letras): 2  |  '2024-06-30' (10 letras): 1  |  '

In [12]:
# Pegando valores anormais
df_datas_estranhas = df_original[df_original['data_venda'] == 'DATA INVALIDA']

df_produtos_tablet = df_original[df_original['produto'] == 'Tablet ']

smartphone = ['Smartphone ', ' Smartphone ']
df_smartphone = df_original[df_original['produto'].isin(smartphone)]

# Puxando a tabela para conferir os valores
print('DATA')
display(df_datas_estranhas)
print('Produtos == Tablet')
display(df_produtos_tablet)
print('Produto == Smartphone')
display(df_smartphone)


DATA


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
16,17,DATA INVALIDA,Cliente_011,Smartphone,Celulares,Centro-Oeste,6.0,2200.0
21,22,DATA INVALIDA,Cliente_023,Smartphone,Celulares,Sudeste,NaN,2200.0
106,107,DATA INVALIDA,Cliente_024,Notebook,Computadores,Nordeste,5.0,3500.0


Produtos == Tablet


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
79,80,2024-11-01,Cliente_016,Tablet,Celulares,Norte,5.0,1800.0


Produto == Smartphone


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
80,81,2024-11-26,Cliente_022,Smartphone,Celulares,Norte,8.0,2200.0
117,118,2024-11-22,Cliente_015,Smartphone,Celulares,Nordeste,8.0,2200.0


In [8]:
# Visualizando as linhas estranhas
df_datas_estranhas = df_original[df_original['data_venda'] == 'DATA INVALIDA']
df_datas_estranhas

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
2,3,DATA INVALIDA,Cliente_026,Mouse,Periféricos,Sul,9.0,120.0
16,17,DATA INVALIDA,Cliente_011,Smartphone,Celulares,Centro-Oeste,6.0,2200.0
21,22,DATA INVALIDA,Cliente_023,Smartphone,Celulares,Sudeste,NaN,2200.0
106,107,DATA INVALIDA,Cliente_024,Notebook,Computadores,Nordeste,5.0,3500.0


Para ficar fácil de interpretar, nós dividimos esses indicadores em três grupos: os de Texto, os de Média/Dispersão, e os de Posição (Quartis).
1) O Grupo dos Textos (Variáveis Categóricas): unique (qauntos valores unicos), top (moda) e freq (nºx moda aparece)
2) O Grupo de Posição (Os Quartis) e o 50% (Mediana)
--> Sempre confie mais na Mediana (50%) para entender o comportamento do seu "cliente padrão".
--> O método estatístico mais famoso do mundo para detectar outliers usa exatamente eles. Chama-se Método IQR (Intervalo Interquartil).

In [9]:
funcoes.resumo_estatistico(df_original)

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
count,150.0,150,150,150,150,150,144.0,147.0
unique,-,113,30,9,3,5,-,-
top,-,2024-05-15,Cliente_024,Notebook,Celulares,Sudeste,-,-
freq,-,5,8,27,53,42,-,-
mean,75.5,-,-,-,-,-,5.42,1596.12
std,43.45,-,-,-,-,-,2.71,1178.31
min,1.0,-,-,-,-,-,1.0,120.0
25%,38.25,-,-,-,-,-,3.0,250.0
50%,75.5,-,-,-,-,-,5.0,1800.0
75%,112.75,-,-,-,-,-,8.0,2200.0
